# 41 — PDF Extraction Agent

This notebook teaches you how to build a LangGraph agent that:

- Downloads a PDF from a URL and extracts raw text using **pdfplumber**
- Uses `with_structured_output(PaperSchema)` to force the LLM to return a **typed Pydantic model**
- Implements a **retry-on-failure loop** using `add_conditional_edges` — the graph routes back to the extract node if validation fails
- Uses a **`retries` counter in state** to prevent infinite loops

By the end you'll understand how to use LLMs as structured parsers over unstructured document content.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q langchain langchain-openai langgraph pdfplumber requests python-dotenv

In [ ]:
import os

if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in Colab secrets or .env"

In [ ]:
# --- Concept Explanation ---
#
# PDF TEXT EXTRACTION: pdfplumber vs alternatives
#   pdfplumber  — precise text/table extraction, handles layout, MIT license
#   PyPDF2      — simpler but less accurate on complex layouts
#   LlamaParse  — cloud-based, handles scanned PDFs via OCR, requires API key
#   Strategy here: pdfplumber for speed + zero external dependencies
#
# STRUCTURED OUTPUT: with_structured_output vs prompt engineering
#   with_structured_output(Schema) — uses function calling / JSON mode under the hood
#     - Guarantees the output matches the Pydantic schema
#     - Raises an exception if the LLM can't fill required fields
#     - Much more reliable than asking the LLM to "return JSON"
#   Prompt engineering ("return JSON with fields...") — fragile, no validation
#
# RETRY PATTERN for unreliable LLM outputs
#   Problem: LLMs sometimes fail to extract fields (bad PDF, ambiguous text)
#   Solution: add_conditional_edges routes back to extract node on failure
#   Guard: retries counter in state prevents infinite loops
#   Pattern: try -> fail? -> increment retries -> retry up to MAX_RETRIES -> give up

print("Approach: download -> text -> structured extraction -> retry on failure")

In [ ]:
import io
from typing import TypedDict

import requests
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel

PDF_URL = "https://arxiv.org/pdf/1706.03762"
MAX_RETRIES = 3
CHARS_LIMIT = 3000


class PaperSchema(BaseModel):
    title: str
    year: int
    key_contribution: str
    architecture_name: str


class ExtractionState(TypedDict):
    url: str
    pdf_text: str
    extracted: dict
    retries: int
    success: bool


def download_pdf_text(url: str) -> str:
    try:
        import pdfplumber
        resp = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
            return " ".join(p.extract_text() or "" for p in pdf.pages[:4])
    except Exception as e:
        return f"[PDF download failed: {e}]"


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
extractor = llm.with_structured_output(PaperSchema)


def download(state: ExtractionState) -> dict:
    if state["pdf_text"]:
        return {}
    text = download_pdf_text(state["url"])
    print(f"  Downloaded {len(text)} chars")
    return {"pdf_text": text[:CHARS_LIMIT]}


def extract(state: ExtractionState) -> dict:
    prompt = (
        f"Extract structured information from this academic paper text:\n\n{state['pdf_text']}\n\n"
        "Fill in: title, year (integer), key_contribution, architecture_name."
    )
    try:
        result = extractor.invoke([HumanMessage(content=prompt)])
        return {"extracted": result.model_dump(), "success": True}
    except Exception as e:
        print(f"  Extraction failed (attempt {state['retries']+1}): {e}")
        return {"extracted": {}, "success": False, "retries": state["retries"] + 1}


def should_retry(state: ExtractionState) -> str:
    if state["success"]:
        return END
    return "extract" if state["retries"] < MAX_RETRIES else END


def create_workflow():
    graph = StateGraph(ExtractionState)
    graph.add_node("download", download)
    graph.add_node("extract", extract)
    graph.set_entry_point("download")
    graph.add_edge("download", "extract")
    graph.add_conditional_edges("extract", should_retry, {"extract": "extract", END: END})
    return graph.compile()


print("All components defined successfully.")

In [ ]:
app = create_workflow()

print(f"Extracting from: {PDF_URL}")
result = app.invoke({
    "url": PDF_URL,
    "pdf_text": "",
    "extracted": {},
    "retries": 0,
    "success": False,
})

if result["success"]:
    print("\nExtracted fields:")
    for k, v in result["extracted"].items():
        print(f"  {k}: {v}")
    print(f"\nRetries used: {result['retries']}")
else:
    print(f"Extraction failed after {result['retries']} retries")

## Exercises

1. **Extend PaperSchema** — add more fields like `abstract: str` and `main_results: str`. Re-run and compare extraction quality.
2. **Process multiple PDFs in parallel** — pass a list of URLs and use Python's `concurrent.futures.ThreadPoolExecutor` to invoke the workflow concurrently.
3. **Add a confidence score** — add `confidence: float` to `PaperSchema` and prompt the LLM to self-report how confident it is in the extraction.
4. **Try a non-academic PDF** — point `PDF_URL` at a technical report or whitepaper and observe how well structured extraction generalises.

## What's Next

- **[13-structured-output](../13-structured-output/)** — structured output fundamentals without the PDF layer
- **[17-corrective-rag](../17-corrective-rag/)** — retry patterns applied to RAG grading and query transformation